In [1]:
import pandas as pd
import numpy as np
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
def get_adj_risk_score():
    results = pd.read_csv("cta_results/pair_results.csv")
    results['response'] = np.where((results['response'] == "Student 1") | (results['response'] == "Student 2"), results['response'], "Removed response")

    rating_counts_df = pd.concat([
    results.loc[results['response'] == 'Student 1', 'id.x'],
    results.loc[results['response'] == 'Student 2', 'id.y']
    ])

    rate_df = rating_counts_df.value_counts().reset_index()
    rate_df.columns = ['id', 'risk_score']

    unknown_counts_df = pd.concat([
    results.loc[results['response'] == 'Removed response', 'id.x'],
    results.loc[results['response'] == 'Removed response', 'id.y']
    ])

    unknown_df = unknown_counts_df.value_counts().reset_index()
    unknown_df.columns = ['id', 'unknown_count']




    cta = pd.read_csv("cta_results/cta_preprocessed_unpaired.csv")
    results_df = pd.merge(pd.merge(cta, rate_df, on = 'id', how = 'left'), unknown_df, on = 'id', how = 'left')
    results_df['risk_score'] = results_df['risk_score'].fillna(0)
    results_df['unknown_count'] = results_df['unknown_count'].fillna(0)
    results_df['risk_score_adj'] = results_df['risk_score']/(results_df['pair_group_size'] - results_df['unknown_count'])

    return results_df

In [3]:
results = get_adj_risk_score()
results.head()

,Unnamed: 0,state,grdlvl,race,sex,spec_speced,spec_gifted,spec_esl,frl,y_yirt,xirt,oob_preds,id,pair_group,pair_group_size,risk_score,unknown_count,risk_score_adj
0,11865,Texas,high school student,"Black non-Hispanic,","They are male,","They are not in special education,","are not in gifted education,",and it is unknown if they are in ESL education.,and whether they recieve free or reduced lunch...,-1.469,-0.921,0.060875,1,1186,10,2.0,0.0,0.2
1,5897,Michigan,high school student,"White non-Hispanic,","They are female,","They are not in special education,","are not in gifted education,",and are not in ESL education.,and whether they recieve free or reduced lunch...,-1.739,-0.937,-0.536193,2,589,10,2.0,0.0,0.2
2,7063,Connecticut,high school student,"Black non-Hispanic,","They are female,","They are not in special education,","are not in gifted education,",and are not in ESL education.,and whether they recieve free or reduced lunch...,0.479,unknown,-0.430060,3,706,10,5.0,0.0,0.5
3,2573,Kentucky,high school student,"White non-Hispanic,","They are female,","They are in special education,","are not in gifted education,",and are not in ESL education.,and whether they recieve free or reduced lunch...,-1.209,-0.76,-0.825887,4,257,10,5.0,0.0,0.5
4,5796,Louisiana,high school student,"Black non-Hispanic,","They are female,","They are not in special education,","are not in gifted education,",and are not in ESL education.,and whether they recieve free or reduced lunch...,0.011,unknown,-0.543447,5,579,10,6.0,0.0,0.6


In [4]:
cta = pd.read_csv("cta_student_level_final.csv")
cta = cta[['state', 'grdlvl', 'race', 'sex', 'spec_speced', 'spec_gifted', 'spec_esl', 'frl', 'y_yirt' ,'xirt', 'treatment']]
cta['id'] = range(1, len(cta) + 1)

results = results[['id', 'oob_preds', 'pair_group', 'pair_group_size', 'risk_score', 'risk_score_adj']]

merged_results = results.merge(cta, on = 'id', how = 'left')

In [5]:
merged_results.head()

,id,oob_preds,pair_group,pair_group_size,risk_score,risk_score_adj,state,grdlvl,race,sex,spec_speced,spec_gifted,spec_esl,frl,y_yirt,xirt,treatment
0,1,0.060875,1186,10,2.0,0.2,TX,H,BLACK NON-HISPANIC,M,0.0,0.0,NaN,1.0,-1.469,-0.921,1
1,2,-0.536193,589,10,2.0,0.2,MI,H,WHITE NON-HISPANIC,F,0.0,0.0,0.0,1.0,-1.739,-0.937,0
2,3,-0.430060,706,10,5.0,0.5,CT,H,BLACK NON-HISPANIC,F,0.0,0.0,0.0,1.0,0.479,NaN,0
3,4,-0.825887,257,10,5.0,0.5,KY,H,WHITE NON-HISPANIC,F,1.0,0.0,0.0,0.0,-1.209,-0.760,0
4,5,-0.543447,579,10,6.0,0.6,LA,H,BLACK NON-HISPANIC,F,0.0,0.0,0.0,1.0,0.011,NaN,0


In [6]:
merged_results.isna().sum()

id                    0
oob_preds             0
pair_group            0
pair_group_size       0
risk_score            0
risk_score_adj        0
state                 0
grdlvl                0
race               1220
sex                 565
spec_speced         210
spec_gifted         210
spec_esl           1581
frl                4298
y_yirt                0
xirt               2748
treatment             0
dtype: int64

In [6]:
# preprocessing for model 

# for categorical variables, replace NAs with "unknown"
merged_results['race'] = merged_results['race'].fillna("Unknown")
merged_results['sex'] = merged_results['sex'].fillna("Unknown")
merged_results['spec_speced'] = merged_results['spec_speced'].fillna("Unknown")
merged_results['spec_gifted'] = merged_results['spec_gifted'].fillna("Unknown")
merged_results['spec_esl'] = merged_results['spec_esl'].fillna("Unknown")
merged_results['frl'] = merged_results['frl'].fillna("Unknown")

# for numeric, mean impute and add another column that captures the missing values 
merged_results["xirt_mis"] = merged_results["xirt"].isna().astype(int)
merged_results['xirt'] = merged_results['xirt'].fillna(merged_results['xirt'].mean())


In [7]:
merged_results.isna().sum()

id                 0
oob_preds          0
pair_group         0
pair_group_size    0
risk_score         0
risk_score_adj     0
state              0
grdlvl             0
race               0
sex                0
spec_speced        0
spec_gifted        0
spec_esl           0
frl                0
y_yirt             0
xirt               0
treatment          0
xirt_mis           0
dtype: int64

In [8]:
# create dummies

X = merged_results[["state", "grdlvl", 'race', 'sex', 'spec_speced', 'spec_gifted', 'spec_esl', 'frl', 'xirt_mis', 'oob_preds', 'xirt', 'y_yirt', 'risk_score_adj', 'treatment']]
df_dummies = pd.get_dummies(X, columns=["state", "grdlvl", 'race', 'sex', 'spec_speced', 'spec_gifted', 'spec_esl', 'frl', 'xirt_mis'], drop_first=True)


In [9]:
df_dummies

,oob_preds,xirt,y_yirt,risk_score_adj,treatment,state_CT,state_KY,state_LA,state_MI,state_NJ,...,sex_Unknown,spec_speced_1.0,spec_speced_Unknown,spec_gifted_1.0,spec_gifted_Unknown,spec_esl_1.0,spec_esl_Unknown,frl_1.0,frl_Unknown,xirt_mis_1
0,0.060875,-0.921000,-1.469,0.2,1,False,False,False,False,False,...,False,False,False,False,False,False,True,True,False,False
1,-0.536193,-0.937000,-1.739,0.2,0,False,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
2,-0.430060,-0.005043,0.479,0.5,0,True,False,False,False,False,...,False,False,False,False,False,False,False,True,False,True
3,-0.825887,-0.760000,-1.209,0.5,0,False,True,False,False,False,...,False,True,False,False,False,False,False,False,False,False
4,-0.543447,-0.005043,0.011,0.6,0,False,False,True,False,False,...,False,False,False,False,False,False,False,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19048,-0.210409,-0.005043,0.077,0.7,0,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,True
19049,1.529610,1.093000,1.297,0.1,1,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
19050,0.894208,1.130000,1.227,0.7,1,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
19051,0.545077,-0.223000,0.620,0.0,1,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [12]:
# poor man's reloop 
import statsmodels.api as sm

# treatment and all covariates 
X1 = df_dummies.drop(['y_yirt','oob_preds', 'risk_score_adj'], axis = 1).astype(float)
X1= sm.add_constant(X1)

X2 = df_dummies.drop(['y_yirt' ,'oob_preds'], axis = 1).astype(float)
X2= sm.add_constant(X2)

y = df_dummies["y_yirt"]

model1 = sm.OLS(y, X1).fit()
model2 = sm.OLS(y, X2).fit()

print([model1.bse['treatment'], model2.bse['treatment']])


[np.float64(0.009925721651965876), np.float64(0.009921645530106623)]


In [21]:
(model1.bse['treatment']**2/model2.bse['treatment']**2) - 1

np.float64(0.0008218312468357691)

In [11]:
df_dummies.columns

Index(['oob_preds', 'xirt', 'y_yirt', 'risk_score_adj', 'treatment',
       'state_CT', 'state_KY', 'state_LA', 'state_MI', 'state_NJ', 'state_TX',
       'grdlvl_M', 'race_ASIAN / PACIFIC ISLANDER', 'race_BLACK NON-HISPANIC',
       'race_HISPANIC', 'race_OTHER RACE / MULTI-RACIAL', 'race_Unknown',
       'race_WHITE NON-HISPANIC', 'sex_M', 'sex_Unknown', 'spec_speced_1.0',
       'spec_speced_Unknown', 'spec_gifted_1.0', 'spec_gifted_Unknown',
       'spec_esl_1.0', 'spec_esl_Unknown', 'frl_1.0', 'frl_Unknown',
       'xirt_mis_1'],
      dtype='object')

In [15]:
# FULL RELOOP

# first without risk score 
X1 = df_dummies.drop(['y_yirt','oob_preds', 'risk_score_adj'], axis = 1).astype(float)
X1= sm.add_constant(X1)

X2 = df_dummies.drop(['y_yirt' ,'oob_preds'], axis = 1).astype(float)
X2= sm.add_constant(X2)

for i in df_dummies.index:
    X_withouti = sm.add_constant(X1.drop(index = i))
    y_withouti = np.delete(y, i)
    model_withouti = sm.OLS(y_withouti, X_withouti).fit()

    observation_it = X1.iloc[[i]]
    observation_it.loc[:,'treatment'] = 1 

    observation_ic = X1.iloc[[i]]
    observation_ic.loc[:,'treatment'] = 0

    #i_t = sm.add_constant(observation_it, has_constant = 'add')
    #i_c = sm.add_constant(observation_ic, has_constant = 'add')

    y_it = model_withouti.predict(observation_it).iloc[0]
    y_ic = model_withouti.predict(observation_ic).iloc[0]
    
    df_dummies.loc[i, 'y_ic_basic'] = y_ic
    df_dummies.loc[i, 'y_it_basic'] = y_it

    X_withouti = sm.add_constant(X2.drop(index = i))
    y_withouti = np.delete(y, i)
    model_withouti = sm.OLS(y_withouti, X_withouti).fit()

    observation_it = X2.iloc[[i]]
    observation_it.loc[:,'treatment'] = 1 

    observation_ic = X2.iloc[[i]]
    observation_ic.loc[:,'treatment'] = 0

    #i_t = sm.add_constant(observation_it, has_constant = 'add')
    #i_c = sm.add_constant(observation_ic, has_constant = 'add')

    y_it = model_withouti.predict(observation_it).iloc[0]
    y_ic = model_withouti.predict(observation_ic).iloc[0]
    
    df_dummies.loc[i, 'y_ic_score'] = y_ic
    df_dummies.loc[i, 'y_it_score'] = y_it


In [16]:
def get_variance(df, p, pred_type = "basic", outcome = "y_yirt", treatment = "treatment"):
    df['m'] = p * df[f'y_it_{pred_type}'] + (1- p) * df[f'y_ic_{pred_type}']

    nc = len(df) - sum(df[f'{treatment}'])
    nt = sum(df[f'{treatment}'])
    N = len(df)

    ec2 = 1/nc * sum((1 - df[f'{treatment}']) * (df[f'y_ic_{pred_type}'] - df[f'{outcome}'])**2)
    et2 = 1/nt * sum(df[f'{treatment}'] * (df[f'y_it_{pred_type}'] - df[f'{outcome}'])**2)

    var_est = 1/N * (p/(1-p) * ec2 + (1-p)/p * et2 + 2 * (ec2 * et2)**0.5)
    return var_est

In [22]:
var_without = get_variance(df_dummies, p = 0.5) # variance without risk score
var_with = get_variance(df_dummies, pred_type = "score", p = 0.5) # variance with risk score 
print(var_without, var_with)

9.261474194126177e-05 9.252240360284412e-05


In [23]:
print(var_without ** 0.5, var_with ** 0.5)

0.009623655331591098 0.009618856668172372


In [20]:
get_variance(df_dummies, p = 0.5)/get_variance(df_dummies, pred_type = "score", p = 0.5) - 1

0.0009980105879439094

In [37]:
import statsmodels.api as sm
# Model 1 - OOB preds + other covariates 

X = df_dummies.drop(['y_yirt', 'risk_score_adj', 'treatment'], axis = 1).astype(float)
X = sm.add_constant(X)
y = df_dummies["y_yirt"]

model_b1 = sm.OLS(y, X).fit()
print(model_b1.summary())

                            OLS Regression Results                            
Dep. Variable:                 y_yirt   R-squared:                       0.468
Model:                            OLS   Adj. R-squared:                  0.467
Method:                 Least Squares   F-statistic:                     669.5
Date:                Tue, 17 Feb 2026   Prob (F-statistic):               0.00
Time:                        15:15:07   Log-Likelihood:                -19080.
No. Observations:               19053   AIC:                         3.821e+04
Df Residuals:                   19027   BIC:                         3.842e+04
Df Model:                          25                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const       

In [38]:
# Model 2 - OOB preds + other covariates + adjusted pair score 

X2 = df_dummies.drop(['y_yirt', 'treatment'], axis = 1).astype(float)
X2 = sm.add_constant(X2)
y = df_dummies["y_yirt"]

model_a1 = sm.OLS(y, X2).fit()
print(model_a1.summary())

                            OLS Regression Results                            
Dep. Variable:                 y_yirt   R-squared:                       0.469
Model:                            OLS   Adj. R-squared:                  0.468
Method:                 Least Squares   F-statistic:                     645.6
Date:                Tue, 17 Feb 2026   Prob (F-statistic):               0.00
Time:                        15:15:09   Log-Likelihood:                -19067.
No. Observations:               19053   AIC:                         3.819e+04
Df Residuals:                   19026   BIC:                         3.840e+04
Df Model:                          26                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const       

In [48]:
from stargazer.stargazer import Stargazer

stargazer = Stargazer([model_a1, model_b1])
print(stargazer.render_latex())

\begin{table}[!htbp] \centering
\begin{tabular}{@{\extracolsep{5pt}}lcc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{2}{c}{\textit{Dependent variable: y_yirt}} \
\cr \cline{2-3}
\\[-1.8ex] & (1) & (2) \\
\hline \\[-1.8ex]
 const & -0.120$^{}$ & -0.046$^{}$ \\
& (0.116) & (0.115) \\
 frl_1.0 & -0.041$^{***}$ & -0.040$^{***}$ \\
& (0.012) & (0.012) \\
 frl_Unknown & -0.052$^{***}$ & -0.052$^{***}$ \\
& (0.017) & (0.017) \\
 grdlvl_M & 0.280$^{***}$ & 0.275$^{***}$ \\
& (0.014) & (0.014) \\
 oob_preds & 0.246$^{***}$ & 0.190$^{***}$ \\
& (0.018) & (0.014) \\
 race_ASIAN / PACIFIC ISLANDER & 0.165$^{}$ & 0.166$^{}$ \\
& (0.116) & (0.116) \\
 race_BLACK NON-HISPANIC & -0.128$^{}$ & -0.130$^{}$ \\
& (0.112) & (0.112) \\
 race_HISPANIC & -0.185$^{}$ & -0.185$^{*}$ \\
& (0.113) & (0.113) \\
 race_OTHER RACE / MULTI-RACIAL & -0.062$^{}$ & -0.066$^{}$ \\
& (0.128) & (0.128) \\
 race_Unknown & -0.076$^{}$ & -0.077$^{}$ \\
& (0.114) & (0.114) \\
 race_WHITE NON-HISPANIC & 0.000$^{}$ & 0.002$

In [39]:
# test between the two models 
model_a1.compare_lr_test(model_b1)

(np.float64(25.368587996905262),
 np.float64(4.7356701061851016e-07),
 np.float64(1.0))

In [40]:
# get leave one out predictions from both models
influence = model_a1.get_influence()
loo_residuals = influence.resid_press
loo_predictions_a1 = df_dummies['y_yirt'] - loo_residuals

influence = model_b1.get_influence()
loo_residuals = influence.resid_press
loo_predictions_b1 = df_dummies['y_yirt'] - loo_residuals

In [41]:
df_dummies['loo_preds_model_a1'] = loo_predictions_a1
df_dummies['loo_preds_model_b1'] = loo_predictions_b1
X = df_dummies[['treatment', 'loo_preds_model_a1']].astype(float)
X = sm.add_constant(X)
y = df_dummies["y_yirt"]

model_a2 = sm.OLS(y, X).fit()
print(model_a2.summary())

                            OLS Regression Results                            
Dep. Variable:                 y_yirt   R-squared:                       0.467
Model:                            OLS   Adj. R-squared:                  0.467
Method:                 Least Squares   F-statistic:                     8353.
Date:                Tue, 17 Feb 2026   Prob (F-statistic):               0.00
Time:                        15:15:15   Log-Likelihood:                -19094.
No. Observations:               19053   AIC:                         3.819e+04
Df Residuals:                   19050   BIC:                         3.822e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -0.0051      0

In [46]:
model_a2.bse

const                 0.006540
treatment             0.009581
loo_preds_model_a1    0.007728
dtype: float64

In [43]:
X2 = df_dummies[['treatment', 'loo_preds_model_b1']].astype(float)
X2 = sm.add_constant(X2)
y = df_dummies["y_yirt"]

model_b2 = sm.OLS(y, X2).fit()
print(model_b2.summary())

                            OLS Regression Results                            
Dep. Variable:                 y_yirt   R-squared:                       0.467
Model:                            OLS   Adj. R-squared:                  0.467
Method:                 Least Squares   F-statistic:                     8331.
Date:                Tue, 17 Feb 2026   Prob (F-statistic):               0.00
Time:                        15:15:21   Log-Likelihood:                -19106.
No. Observations:               19053   AIC:                         3.822e+04
Df Residuals:                   19050   BIC:                         3.824e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -0.0052      0

In [47]:
model_b2.bse # smaller, but barely 

const                 0.006544
treatment             0.009587
loo_preds_model_b1    0.007738
dtype: float64

In [49]:
stargazer = Stargazer([model_a2, model_b2])
print(stargazer.render_latex())

\begin{table}[!htbp] \centering
\begin{tabular}{@{\extracolsep{5pt}}lcc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{2}{c}{\textit{Dependent variable: y_yirt}} \
\cr \cline{2-3}
\\[-1.8ex] & (1) & (2) \\
\hline \\[-1.8ex]
 const & -0.005$^{}$ & -0.005$^{}$ \\
& (0.007) & (0.007) \\
 loo_preds_model_a1 & 0.999$^{***}$ & \\
& (0.008) & \\
 loo_preds_model_b1 & & 0.999$^{***}$ \\
& & (0.008) \\
 treatment & 0.011$^{}$ & 0.011$^{}$ \\
& (0.010) & (0.010) \\
\hline \\[-1.8ex]
 Observations & 19053 & 19053 \\
 $R^2$ & 0.467 & 0.467 \\
 Adjusted $R^2$ & 0.467 & 0.467 \\
 Residual Std. Error & 0.659 (df=19050) & 0.660 (df=19050) \\
 F Statistic & 8352.819$^{***}$ (df=2; 19050) & 8330.940$^{***}$ (df=2; 19050) \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{2}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
